# **5. DistilBERT**

**DistilBERT** es un modelo de lenguaje basado en la arquitectura Transformer, entrenado mediante knowledge distillation a partir de BERT.

Recibe:

- Una secuencia de tokens numéricos que representan el texto. 
- Una máscara binaria de tokens atentidos e ignorados
- Labels de clasificación

Estas entradas se generan usando el tokenizador     `_DistilBertTokenizerFast_`  , que convierte texto en tensores compatibles con el modelo.
  



In [1]:
# CONFIGURACIÓN INICIAL
# =========================

import os, time, ast, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)

from transformers import (
    get_scheduler,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)



Using device: cpu


In [2]:
# MODULO DE MANEJO DE DATOS
# ---------------------------------

#  [Carga de datos]
df = pd.read_json('https://raw.githubusercontent.com/Darally06/NLP-Jarvis-Hiring/refs/heads/main/Data/clean_words.json', lines=True)
df = df.rename(columns={'macro_label': 'Category'})
print(f"Registros totales: {len(df)}")

# Codificar etiquetas
le = LabelEncoder()
df["label"] = le.fit_transform(df["Category"])

# Dividir dataset
train, temp = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df["label"])
val, test = train_test_split(temp, test_size=0.5, random_state=SEED, stratify=temp["label"])

print(f"Distribución de los datos: \nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Registros totales: 2483
Distribución de los datos: 
Train: 1738 | Val: 372 | Test: 373


## 🔹Modelo

In [3]:
# DEFINIR TEXTOS Y ETIQUETAS

train_texts = train["clean_text"]
train_labels = train["label"]

val_texts = val["clean_text"]
val_labels = val["label"]

test_texts = test["clean_text"]
test_labels = test["label"]


In [4]:
# Tokenización y codificación de etiquetas
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenización
def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        return_tensors='pt'
    )


train_texts = train_texts.sample(frac=0.5, random_state=42)
train_labels = train_labels.loc[train_texts.index]

val_texts = val_texts.sample(frac=0.5, random_state=42)
val_labels = val_labels.loc[val_texts.index]

test_texts = test_texts.sample(frac=0.5, random_state=42)
test_labels = test_labels.loc[test_texts.index]

train_encodings = tokenize_texts(train_texts)
val_encodings = tokenize_texts(val_texts)
test_encodings = tokenize_texts(test_texts)

In [6]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

num_labels = df['label'].nunique()

# Modelo
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',  # o distilbert-base-uncased si es en inglés
    num_labels=num_labels
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Se define una clase `Dataset`  personalizada para estructurar los datos tokenizados y sus etiquetas. Esto permite crear objetos compatibles con los DataLoader de PyTorch. Luego se instancian los datasets de entrenamiento, validación y prueba. Finalmente, se carga el modelo **DistilBERT** base con una capa de clasificación ajustada al número de clases (`num_labels`).

In [7]:
# Función de métricas personalizada
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    prec = precision_score(labels, preds, average='weighted')
    rec = recall_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


Se define la función `compute_metrics` para evaluar el rendimiento del modelo en cada época, calculando **accuracy**, **F1-score**, **precisión** y **recall ponderados**.

En los `TrainingArguments`, se establecen los hiperparámetros, comunes para fine-tuning de modelos basados en Transformers:

* **learning_rate = 5e-5**, un valor estándar y estable para evitar sobreajuste en modelos preentrenados;
* **batch_size = 16**, balance entre eficiencia y uso de memoria GPU;
* **num_train_epochs = 3**, suficiente para que el modelo converja sin sobreentrenarse;
* **weight_decay = 0.01**, para regularización ligera;
* **eval/save/logging_strategy = 'epoch'**, con el fin de monitorear métricas y guardar el mejor modelo tras cada época;
* **load_best_model_at_end=True**, asegurando que se retenga el modelo con mayor precisión.

Finalmente, el objeto `Trainer` centraliza el proceso de entrenamiento y validación.


## 🔹 Entrenamiento

In [8]:
# Entrenamiento
start_time = time.time()
trainer.train()
end_time = time.time()
train_time = end_time - start_time
print(f"\n Tiempo total de entrenamiento: {train_time:.2f} segundos")

save_path = "./distilbert_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Modelo guardado en: {save_path}")

c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.437700,1.072763,0.596774,0.567627,0.593688,0.596774
2,0.901500,0.645889,0.790323,0.789531,0.794811,0.790323
3,0.593800,0.560765,0.844086,0.842128,0.846550,0.844086


c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



 Tiempo total de entrenamiento: 25816.38 segundos
Modelo guardado en: ./distilbert_model


Durante el entrenamiento se observa una **mejora progresiva y consistente** en las métricas a lo largo de las tres épocas. La **pérdida de entrenamiento** desciende de *1.43* a *0.59*, y la **pérdida de validación** disminuye de *1.07* a *0.56*, lo que indica una buena capacidad de aprendizaje sin señales de sobreajuste.

El modelo alcanza una **precisión final (accuracy)** de **0.84** y un **F1-score de 0.84**, valores muy cercanos entre sí, lo cual refleja un equilibrio adecuado entre precisión y exhaustividad. Las métricas de **precisión (0.85)** y **recall (0.84)** confirman que el modelo logra reconocer correctamente la mayoría de las clases sin sacrificar la consistencia en las predicciones.

En conjunto, los resultados sugieren que el entrenamiento fue exitoso: el modelo convergió de forma estable y alcanzó un rendimiento alto en validación, lo que evidencia una **buena capacidad de generalización** tras solo tres épocas de ajuste fino.


## 🔹 Evaluación de rendimiento

In [9]:
# Validación
val_metrics = trainer.evaluate(val_dataset)
val_acc = val_metrics.get('eval_accuracy', None)
print(f"\Accuracy de validación: {val_acc:.4f}")

# Prueba
test_metrics = trainer.evaluate(test_dataset)
test_acc = test_metrics.get('eval_accuracy', None)
print(f" Accuracy de prueba: {test_acc:.4f}")

# 4. Predicciones y reporte de clasificación
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

# Reporte
report = classification_report(
    true_labels,
    preds,
    target_names=le.classes_,
    digits=4
)

print("\n Classification Report:\n")
print(report)
report_path = "C:/DeepLearning/DL_Proyecto_3/classification_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("DistilBERT Classification Report\n")
    f.write(f"Tiempo de entrenamiento: {train_time:.2f} segundos\n")
    f.write(f"Accuracy validación: {val_acc:.4f}\n")
    f.write(f"Accuracy prueba: {test_acc:.4f}\n\n")
    f.write(report)

print(f" Reporte guardado en: {report_path}")

c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


\Accuracy de validación: 0.8441


c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


 Accuracy de prueba: 0.8065


c:\Users\TAWTOCA\anaconda3\envs\machine\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



 Classification Report:

                                    precision    recall  f1-score   support

          Creatividad y Producción     0.7778    0.7778    0.7778        27
               Negocios y Finanzas     0.9149    0.8958    0.9053        48
            Otros y Especializados     0.7568    0.7000    0.7273        40
Servicios Profesionales y Públicos     0.6667    0.7222    0.6933        36
              Tecnología y Ciencia     0.8889    0.9143    0.9014        35

                          accuracy                         0.8065       186
                         macro avg     0.8010    0.8020    0.8010       186
                      weighted avg     0.8080    0.8065    0.8067       186

 Reporte guardado en: C:/DeepLearning/DL_Proyecto_3/classification_report.txt


El modelo logra un **excelente rendimiento general**, con una **exactitud de validación del 84.4 %** y una **exactitud de prueba del 80.6 %**, lo que indica que mantiene un desempeño sólido al enfrentarse a datos no vistos, con mínima pérdida de generalización.

El **reporte de clasificación** muestra un comportamiento equilibrado entre las distintas categorías, sin clases dominantes que distorsionen las métricas. Las clases con mejor desempeño son:

* **Negocios y Finanzas (F1 = 0.91)** y **Tecnología y Ciencia (F1 = 0.90)**, donde el modelo identifica correctamente la mayoría de las instancias.

Mientras que categorías más heterogéneas como **Servicios Profesionales y Públicos (F1 = 0.69)** y **Otros y Especializados (F1 = 0.73)** presentan una leve disminución en recall, lo que sugiere una mayor confusión con clases vecinas en el espacio semántico.

El **promedio macro (F1 = 0.80)** y el **promedio ponderado (F1 = 0.81)** confirman que el modelo mantiene un rendimiento equilibrado entre clases grandes y pequeñas.
En conjunto, los resultados evidencian que el **DistilBERT fine-tuned** logró una **representación robusta del texto y una clasificación precisa**, con un tiempo de entrenamiento total de aproximadamente **7 horas y 10 minutos**.
